In [ ]:
import os
import datetime
import instructor
import ollama
import stanza
from pydantic import BaseModel, Field

In [ ]:
import sys
import torch
import numpy as np
from importlib import reload

# Intercept and patch the global torch load function
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

# Reload stanza if it has already been initialized in this session
if 'stanza' in sys.modules:
    reload(sys.modules['stanza'])
    
# Tell PyTorch's unpickler that NumPy's multiarray reconstructor is safe to load
torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.dtype
])

import spacy_stanza
import random
stanza.download("he")

import json
from jsonschema import validate, ValidationError

In [ ]:
def save_ai_story(prompt, filepath, model_name='llama3.1'):
    dir_name = os.path.dirname(filepath)
    if dir_name and not os.path.exists(dir_name):
        os.makedirs(dir_name)
        print(f"Created missing directory structure: {dir_name}")

    print(f"Querying Ollama ({model_name}) for: {os.path.basename(filepath)}...")
    
    # Pass the targeted model_name dynamically here
    response = ollama.generate(model=model_name, prompt=prompt)
    story_text = response['response']
    
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(story_text)
        
    print(f"Successfully generated and saved story to: {filepath}")
    return story_text 

In [ ]:
def save_ai_story_he(prompt, filepath, model_name='aminadaven/dictalm2.0-instruct:q4_k_m'):
    dir_name = os.path.dirname(filepath)

    if dir_name and not os.path.exists(dir_name):
        os.makedirs(dir_name)
        print(f"Created missing directory structure: {dir_name}")

    print(f"Querying Ollama ({model_name}) for: {os.path.basename(filepath)}...")

    response = ollama.generate(model=model_name, prompt=prompt)
    story_text = response['response']

    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(story_text)

    print(f"Successfully generated and saved story to: {filepath}")

    return story_text

# Story Generator

Generates a 200-300 word story in English and Hebrew to be used for the quests and quizzes.

## Steps

1. English Narrative Generation via Llama
1. Hebrew Translation Optimization via DictaLM

In [ ]:
# Core story elements
level = "CEFR A1/A2"
character_name = "{{pet_name}}"
animal_type = "faun"
gender = "Male"
quest_name = "garden_adventure"

# Story guidance variables
story_title = "יום הגינון של פאון (Faun's Gardening Day)"
story_overview = (
    "Faun has a strong רָעָב (hunger) for a sweet, crunchy treat. "
    "He decides to become a gardener for a day, taking a סַל (basket) "
    "of tools out to the sunny גַּן (garden). He carefully buries a small "
    "זֶרַע (seed) in the dirt, waters it, and proudly harvests his very "
    "first home-grown גֶּזֶר (carrot)."
)

# Python list of vocabulary words
word_list = ["רעב", "סל", "גן", "זרע", "גזר"]

# Format the word list for the prompt
formatted_words = "\n".join([f"- {word}" for word in word_list])

# English Generation Prompt
large_prompt_en = f"""
You are an expert children's book author. Write an original, charming short story in simple, clear English based on this plot overview: 
"{story_overview}"

Character Profiles & Genders:
1. Protagonist: "{character_name}" (the pet). 
   - CRITICAL: "{character_name}"'s gender is dynamic. For this generation run, "{character_name}" is "{gender}". 
   All verbs, pronouns, and adjectives describing "{character_name}"'s actions must strictly conform to grammar rules.
2. Auxiliary Character: "Adva" (the gardener).
   - GENDER: Static Female. All verbs, pronouns, and adjectives describing Adva's actions must ALWAYS 
   remain in the feminine singular (e.g., הלכה, אמרה, שלה). This must NOT change regardless of the Protagonist's active gender.

Rules:
- Keep the sentences short and clear so they translate cleanly into early-intermediate language structures.
- Do not create any other names, use nouns only (bird, girl, city, etc).
- Do not explain grammar or add any extra text to the story, aside from the story text itself.
- Do not format any text in the story aside from using new lines.
"""

# Generate the unique timestamped filenames to prevent overriding
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
animal_folder = animal_type.lower().replace(" ", "_")
filename = f"{timestamp}_{quest_name}"

# Generate the English story using llama3.1
english_story_output = save_ai_story(
    prompt=large_prompt_en,
    filepath=f"./quests/{animal_folder}/{filename}_en.md",
    model_name='llama3.1'
)


In [ ]:
def make_gender_prompt(base_story, gender):
    return f"""
- Rewrite this story using {gender} grammatical agreement throughout.  

Story:
{base_story}
"""
    
# Construct the dynamic Hebrew translation prompt, feeding it the English text
large_prompt_he = f"""
You are an expert bilingual translator fluent in both English and Hebrew. 

Translate the following English story into grammatically correct, natural Hebrew suitable for a CEFR A1/A2 language learner. 

English Story to Translate:
\"\"\"
{english_story_output}
\"\"\"

Rules for the translation:
- Do NOT use vowel points (nikkud) at all. Write in clean, modern, unpointed Hebrew text (Ktav Male).
- Ensure strict gender agreement (since the main character {character_name} is {gender}, use proper masculine verb inflections and adjectives).
- Match the key concepts exactly to these Hebrew vocabulary items:
{formatted_words}
- Do not explain grammar, do not add conversational notes, and do not include the original English in your output. Return ONLY the Hebrew translation.

CRITICAL OUTPUT FORMATTING RULES:
- Return structured output only using a direct Hebrew translation.
- DO NOT include introductory remarks like "Here is the translation...".
- DO NOT include conversational text, pleasantries, or explanations.
- Start your response directly with the translated Hebrew title or the first line of the translated story text.
- Match the new lines from the English version.
- Do not let the gender of the protagonist influence or mutate the grammatical gender of other characters. 
  Keep their linguistic "coreference chains" completely separate and independent.
"""

# Pass the dynamic translation prompt specifically to DictaLM, which has been built specifically for Hebrew
hebrew_story_output = save_ai_story_he(
    prompt=large_prompt_he,
    filepath=f"./quests/{animal_folder}/{filename}_he.md",
    model_name='aminadaven/dictalm2.0-instruct:q4_k_m'
);


In [ ]:

story_text_he = {"male": "", "female": "", "neutral": ""}

story_text_he["male"] = hebrew_story_output
story_text_he["neutral"] = hebrew_story_output
story_text_he["female"] = save_ai_story_he(
    prompt=make_gender_prompt(hebrew_story_output, 'female'),
    filepath=f"./quests/{animal_folder}/{filename}_he.md",
    model_name='aminadaven/dictalm2.0-instruct:q4_k_m'
);

## Generate Quiz

In [ ]:
# 2. Initialize the Stanza Hebrew pipeline wrapper
nlp = spacy_stanza.load_pipeline("he")

In [ ]:
def extract_story_sentences(story_hebrew, max_sentences=20):
    """
    Build aligned story sentence objects for male/female/neutral variants.
    """

    docs = {
        "male": list(nlp(story_hebrew["male"]).sents),
        "female": list(nlp(story_hebrew["female"]).sents),
        "neutral": list(nlp(story_hebrew["neutral"]).sents)
    }

    sentence_count = min(
        len(docs["male"]),
        len(docs["female"]),
        len(docs["neutral"])
    )

    story_sentences = []

    for idx in range(sentence_count):
        male_sent = docs["male"][idx].text.strip()
        female_sent = docs["female"][idx].text.strip()
        neutral_sent = docs["neutral"][idx].text.strip()

        if len(neutral_sent.split()) < 2:
            continue

        sentence_id = f"s{len(story_sentences) + 1}"
        story_sentences.append({
            "id": sentence_id,
            "male": male_sent,
            "female": female_sent,
            "neutral": neutral_sent
        })

        if len(story_sentences) >= max_sentences:
            break

    return story_sentences

In [ ]:
def extract_cloze_quizzes(story_hebrew, target_words, story_sentences):
    """
    Create cloze quizzes anchored to story_sentences with sentence_id links.
    """

    docs = {
        "male": nlp(story_hebrew["male"]),
        "female": nlp(story_hebrew["female"]),
        "neutral": nlp(story_hebrew["neutral"])
    }

    cloze_quizzes = []

    # Build shared POS bank from all variants.
    pos_bank = {}
    for doc in docs.values():
        for token in doc:
            if token.is_alpha and not token.is_stop:
                pos_bank.setdefault(token.pos_, set()).add(token.lemma_)

    for sentence_variants in story_sentences:
        neutral_sentence = sentence_variants.get("neutral", "")
        neutral_doc = nlp(neutral_sentence)
        sentence_id = sentence_variants.get("id")

        if not sentence_id:
            continue

        for target in target_words:
            target_token = next(
                (
                    t for t in neutral_doc
                    if t.text == target or t.lemma_ == target
                ),
                None
            )

            if not target_token:
                continue

            prompts = {}

            for gender in ("male", "female", "neutral"):
                gender_sentence = sentence_variants.get(gender, sentence_variants.get("neutral", ""))
                gender_doc = nlp(gender_sentence)

                matching_token = next(
                    (
                        t for t in gender_doc
                        if t.lemma_ == target_token.lemma_ or t.text == target_token.text
                    ),
                    None
                )

                if matching_token:
                    blank_text = gender_sentence.replace(matching_token.text, "_____", 1)
                else:
                    blank_text = gender_sentence

                prompts[gender] = blank_text

            target_pos = target_token.pos_
            distractor_pool = list(pos_bank.get(target_pos, set()))
            distractor_pool = [
                d for d in distractor_pool
                if d != target_token.text and d != target_token.lemma_ and d != target
            ]

            fallbacks = ["בית", "חבר", "מקום", "ילד"]
            while len(distractor_pool) < 3:
                distractor_pool.append(random.choice(fallbacks))

            distractors = random.sample(distractor_pool, 3)
            options = distractors + [target_token.text]
            random.shuffle(options)

            cloze_quizzes.append({
                "type": "cloze",
                "sentence_id": sentence_id,
                "prompt": prompts,
                "answer": target_token.text,
                "options": options
            })

            break

    return cloze_quizzes

In [ ]:
# 1. Define the distantlife quest schema to catch errors before deployment
QUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "meta": {
            "type": "object",
            "properties": {
                "schema_version": {"type": "string", "const": "1.0.0"},
                "generator": {"type": "string", "const": "quest_pipeline_v1"},
                "generated_at": {"type": "string"}
            },
            "required": ["schema_version", "generator", "generated_at"]
        },
        "quest_id": {"type": "string"},
        "locale": {"type": "string"},
        "theme": {"type": "string"},
        "quest_type": {"type": "string"},
        "quest_line_id": {"type": "string"},
        "allowed_pet_type_ids": {
            "type": "array",
            "items": {"type": "integer"}
        },
        "unlock_cost": {"type": "integer"},
        "review_status": {"type": "string"},
        "episodes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "episode_id": {"type": "string"},
                    "title": {"type": "string"},
                    "story_sentences": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "id": {"type": "string"},
                                "male": {"type": "string"},
                                "female": {"type": "string"},
                                "neutral": {"type": "string"}
                            },
                            "required": ["id", "male", "female", "neutral"]
                        }
                    },
                    "quiz": {
                        "type": "object",
                        "properties": {
                            "questions": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "type": {"type": "string"},
                                        "sentence_id": {"type": "string"},
                                        "answer": {"type": "string"},
                                        "options": {"type": "array"}
                                    },
                                    "required": ["type"]
                                }
                            }
                        },
                        "required": ["questions"]
                    }
                },
                "required": ["episode_id", "title", "story_sentences", "quiz"]
            }
        }
    },
    "required": [
        "meta",
        "quest_id",
        "locale",
        "theme",
        "quest_type",
        "quest_line_id",
        "allowed_pet_type_ids",
        "episodes"
    ]
}

In [ ]:
# Raw inputs from your Ollama story generation step
story_hebrew_raw = story_text_he
story_english_raw = english_story_output
vocab_targets = word_list

# Build aligned sentence structure and quizzes
story_sentences = extract_story_sentences(story_hebrew_raw)
quizzes = extract_cloze_quizzes(story_hebrew_raw, vocab_targets, story_sentences)

In [ ]:
# Package into the canonical distantlife quest structure
quest_payload = {
    "meta": {
        "schema_version": "1.0.0",
        "generator": "quest_pipeline_v1",
        "generated_at": datetime.datetime.utcnow().isoformat()
    },
    "quest_id": "hungry_carrot_01",
    "locale": "he",
    "theme": "vegetables",
    "quest_type": "story_quest",
    "quest_line_id": "hungry_carrot",
    "allowed_pet_type_ids": [10],
    "unlock_cost": 50,
    "review_status": "approved",
    "episodes": [
        {
            "episode_id": "hungry_carrot_01_ep1",
            "title": "הסל הריק",
            "story_sentences": story_sentences,
            "quiz": {
                "questions": quizzes
            }
        }
    ]
}

# Validate and save
try:
    validate(instance=quest_payload, schema=QUEST_SCHEMA)
    print("JSON validation passed! Enforcing schema alignment.")

    output_path = "./quests/he/hungry_carrot_01.json"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(quest_payload, f, ensure_ascii=False, indent=2)

    print(f"Success! Quest file written directly to assets folder: {output_path}")

except ValidationError as e:
    print(f"Schema violation detected: {e.message}")